# **Word2Vec: a Prediction-Based Method**

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# 📖 TABLE OF CONTENTS

- [Setup](#Setup)
- [Idea](#Idea)
- [Objective Function: Negative Log-Likelihood](#Objective-Function:-Negative-Log-Likelihood)
- [How to Train: by Gradient Descent, One Word at a Time](#How-to-Train:-by-Gradient-Descent,-One-Word-at-a-Time)
- [References](#References)

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# Setup

Before we jump in, let's quickly run the setup cell. It checks the environment (GPU, library versions) and sets random seeds for reproducibility — we want identical results on every run.

In [ ]:
# 🔧 Setup: Run this cell first!
# Environment check + random seeds for reproducibility

import sys
import time
import random
import numpy as np
import pandas as pd
import matplotlib as mpl
import torch
import torch.nn as nn

print(f"📦 Python       {sys.version.split()[0]}")
print(f"🔢 NumPy        {np.__version__}")
print(f"🐼 Pandas       {pd.__version__}")
print(f"📊 Matplotlib   {mpl.__version__}")
print(f"🔥 PyTorch      {torch.__version__}")

# GPU check (only needed for optional deep-feature experiments)
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    device = torch.device("cpu")
    print("⚠️ No GPU detected")

DEVICE_LABEL = "GPU" if device.type == "cuda" else "CPU"

# Random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"\n🎲 Random seed set to {SEED} (Python, NumPy, PyTorch)")

%matplotlib inline
%config InlineBackend.figure_format = 'jpeg'   # photos as JPEG, not PNG — ~10× smaller notebook
# Remove character limits for all columns of Pandas dataframes permanently in this session
pd.set_option('display.max_colwidth', None)

📦 Python       3.13.15
🔢 NumPy        2.1.3
🐼 Pandas       2.2.3
📊 Matplotlib   3.10.0
🔥 PyTorch      2.11.0+cpu
⚠️ No GPU detected

🎲 Random seed set to 42 (Python, NumPy, PyTorch)


![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# Idea

Let us remember our main idea again:

**<u>Main idea</u>: We have to put information about contexts into word vectors**.

While count-based methods took this idea quite literally, Word2Vec uses it in a different manner:

**<u>How</u>: Learn word vectors by teaching them to predict contexts**.

Word2Vec is a model whose parameters are word vectors. These parameters are optimized iteratively for a certain objective. The objective forces word vectors to "know" contexts a word can appear in: the vectors are trained to predict possible contexts of the corresponding words. As you remember from the distributional hypothesis, if vectors "know" about contexts, they "know" word meaning.

**<u>Learned Parameters</u>**: word vectors

**<u>Goal</u>**: make each vector "know" about the contexts of its word

**<u>How</u>**: train vectors to predict possible contexts from words (or, alternatively, words from contexts)

Word2Vec is an iterative method. Its main idea is as follows:
* take a huge text corpus;
* go over the text with a sliding window, moving one word at a time. At each step, there is a central word and context words (other words in this window);
* for the central word, compute probabilities of context words;
* adjust the vectors to increase these probabilities.

<p float="left">
  <img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Word2Vec-01.png" width="49%" />
  <img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Word2Vec-02.png" width="49%" />
</p>

<p float="left">
  <img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Word2Vec-03.png" width="49%" />
  <img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Word2Vec-04.png" width="49%" />
</p>

<p float="left">
  <img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Word2Vec-05.png" width="49%" />
  <img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Word2Vec-06.png" width="49%" />
</p>

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# Objective Function: Negative Log-Likelihood

For each position $t = 1, 2, ...,T$ in a text corpus, Word2Vec predicts context words within an m-sized window given the central word $\color{#88bd33}{w_t}$:

\begin{equation*}
\color{#88bd33}{\mbox{Likelihood}} \color{black}= L(\theta)=
    \prod\limits_{t=1}^T\prod\limits_{-m\le j \le m, j\neq 0}P(\color{#888}{w_{t+j}}|\color{#88bd33}{w_t}\color{black}, \theta),
\end{equation*}

where $\theta$ are all variables to be optimized. The objective function (aka loss function or cost function) $J(\theta)$ is the average negative log-likelihood:

\begin{equation*}
\color{#88bd33}{\mbox{Loss}} \color{black}= J(\theta) = -\frac{1}{T} \log {L(\theta)} = -\frac {1}{T} \sum_{t=1}^{T} \sum_{\substack{-m \le j \le m \\ j \neq 0}}
 \log P(\color{#888}{w_{t+j}}|\color{#88bd33}{w_t}\color{black}, \theta)
\end{equation*}

Note how well the loss agrees with our main idea above: go over text with a sliding window and compute probabilities. Now let's find out how to compute these probabilities.

**How to calculate $P(\color{#888}{w_{t+j}}\color{black}|\color{#88bd33}{w_t}\color{black}, \theta) ?$**

For each word $w$ we will have two vectors:

* $\color{#88bd33}{v_w}$ when it is a $\color{#88bd33}{\text{central word}}$;
* $\color{#888}{u_w}$ when it is a $\color{#888}{\text{context word}}$.

(Once the vectors are trained, usually we throw away context vectors and use only word vectors.)

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Word2Vec-07.png" width="300">

Then for the central word $\color{#88bd33}{c}$ (c - central) and the context word $\color{#888}{o}$ (o - outside word), the probability of the context word is given below:

\begin{equation*}
P(\color{#888}{o}|\color{#88bd33}{c}) = \frac {exp(\color{#888}{u_o^T} \cdot \color{#88bd33}{v_c})}{\Sigma_{w \in V} exp(\color{#888}{u_w^T} \cdot \color{#88bd33}{v_c})}
\end{equation*}

where:
* $\color{#888}{u_o^T} \cdot \color{#88bd33}{v_c}$ dot product measures the similarity of $\color{#888}{o}$ and $\color{#88bd33}{c}$. Larger dot product => larger probability
* $\Sigma_{w \in V}$ normalizes over entire vocabulary to get probability distribution

**Note**: this is the softmax function!

The function above is an example of the **softmax function**:

\begin{equation*}
\text{softmax}(x_i) = \frac {exp(x_i)}{\sum_{j=i}^n exp(x_j)}
\end{equation*}

**Softmax** maps arbitrary values $x_i$ to a probability distribution $p_i$:

* "**max**" because the largest $x_i$ will have the largest probability $p_i$;
* "**soft**" because all probabilities are non-zero.

You will deal with this function quite a lot over the NLP course (and in Deep Learning in general).

Now, we can go over the illustration. Note that for $\color{#88bd33}{\text{central words}}$ and $\color{#888}{\text{context words}}$, different vectors are used. For example, first the word `a` is central and we use $\color{#88bd33}{v_a}$, but when it becomes context, we use $\color{#888}{u_a}$ instead.

<p float="left">
  <img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Word2Vec-08.png" width="49%" />
  <img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Word2Vec-09.png" width="49%" />
</p>

<p float="left">
  <img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Word2Vec-10.png" width="49%" />
  <img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Word2Vec-11.png" width="49%" />
</p>

<p float="left">
  <img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Word2Vec-12.png" width="49%" />
  <img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Word2Vec-13.png" width="49%" />
</p>

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# How to Train: by Gradient Descent, One Word at a Time

Let us recall that our parameters $\theta$ are vectors $\color{#88bd33}{v_w}$ and $\color{#888}{u_w}$ for all words in the vocabulary. These vectors are learned by optimizing the training objective via gradient descent (with some learning rate $\alpha$):

\begin{equation*}
\theta^{new} = \theta^{old} - \alpha \nabla_{\theta} J(\theta)
\end{equation*}

**<u>One word at a time</u>**

We make these updates one at a time: each update is for a single pair of a center word and one of its context words. Look again at the loss function:

\begin{equation*}
\color{#88bd33}{\mbox{Loss}} \color{black}= J(\theta) = -\frac{1}{T} \log {L(\theta)} = -\frac {1}{T} \sum_{t=1}^{T} \sum_{\substack{-m \le j \le m \\ j \neq 0}}
 \log P(\color{#888}{w_{t+j}}|\color{#88bd33}{w_t}\color{black}, \theta) = \frac {1}{T} \sum_{t=1}^{T} \sum_{\substack{-m \le j \le m \\ j \neq 0}} J_{t,j}(\theta).
\end{equation*}

For the center word $\color{#88bd33}{w_t}$, the loss contains a distinct term $J_{t,j}(\theta) = -\log P(\color{#888}{w_{t+j}}|\color{#88bd33}{w_t}\color{black}, \theta)$ for each of its context words $\color{#888}{w_{t+j}}$. Let us look in more detail at just this one term and try to understand how to make an update for this step. For example, let's imagine we have a sentence:

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Word2Vec-14.png" width="400" />

with the central word $\color{#88bd33}{\text{cat}}$, and four context words. Since we are going to look at just one step, we will pick only one of the context words; for example, let's take $\color{#888}{\text{cute}}$. Then the loss term for the central word $\color{#88bd33}{\text{cat}}$ and the context word $\color{#888}{\text{cute}}$ is:

\begin{equation*}
J_{t,j}(\theta)= -\log P(\color{#888}{cute}\color{black}|\color{#88bd33}{cat}\color{black}) =
        -\log \frac{\exp\color{#888}{u_{cute}^T}\color{#88bd33}{v_{cat}}}{
       \sum\limits_{w\in Voc}\exp{\color{#888}{u_w^T}\color{#88bd33}{v_{cat}} }} =
    -\color{#888}{u_{cute}^T}\color{#88bd33}{v_{cat}}\color{black}
        + \log \sum\limits_{w \in Voc}\exp{\color{#888}{u_w^T}\color{#88bd33}{v_{cat}}}\color{black}{.}
\end{equation*}

Note which parameters are present at this step:

* from vectors for $\color{#88bd33}{\text{central words}}$, only $\color{#88bd33}{v_{cat}}$;
* from vectors for $\color{#888}{\text{context words}}$, all $\color{#888}{v_w}$ (for all words in the vocabulary).

Only these parameters will be updated at the current step.

Below is the schematic illustration of the derivations for this step.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Word2Vec-15.png" width="600" />

By making an update to minimize $J_{t,j}(\theta)$, we force the parameters to **<u>increase</u>** similarity (dot product) of $\color{#88bd33}{v_{cat}}$ and $\color{#888}{u_{cute}}$ and, at the same time, to **<u>decrease</u>** similarity between $\color{#88bd33}{v_{cat}}$ and $\color{#888}{u_{w}}$ for all other words $w$ in the vocabulary.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Word2Vec-16.png" width="400" />

This may sound a bit strange: why do we want to decrease similarity between $\color{#88bd33}{v_{cat}}$ and all other words, if some of them are also valid context words (e.g., $\color{#888}{\text{grey}}$, $\color{#888}{\text{playing}}$, $\color{#888}{\text{in}}$, on our example sentence)?

But do not worry: since we make updates for each context word (and for all central words in your text), <u>on average over all updates</u> our vectors will learn the distribution of the possible contexts.

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# References

- **[NLP Course by Lena Voita](https://lena-voita.github.io/nlp_course.html)**
- **Key Papers:**
    - [Word2Vec Paper, 2013](https://arxiv.org/pdf/1301.3781)

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Word_Embeddings-01.png" width="400">

<table>
  <tr>
    <td align="center">
      <img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Word_Embeddings-01.png" width="400">
    </td>
  </tr>
  <tr>
    <td align="center">
      <em>Figure 1: Word Embeddings</em>
    </td>
  </tr>
</table>

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)